In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_53.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_62.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_56.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_71.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_15.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_99.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_80.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_42.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_26.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_21.pdf
/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/SAP Develope

In [2]:
# cell 1
!pip install -q sentence-transformers pdfplumber pytesseract pdf2image pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 68.7 MB/s eta 0:00:00:00:01


In [3]:
# cell 2
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 103 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (4,135 kB/s)       
Selecting previously unselected package poppler-utils.
(Reading database ... 120968 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-u

In [4]:
# cell 3
import os
import pandas as pd
import numpy as np

from pypdf import PdfReader
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
#cell 4
resume_folder = "/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF"

print("Exists:", os.path.exists(resume_folder))

Exists: True


In [6]:
#cell 5
import os

def extract_text_from_pdf(pdf_path):

    text = ""

    try:

        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                extracted = page.extract_text()

                if extracted:
                    text += extracted + " "

        if len(text.strip()) < 50:

            images = convert_from_path(pdf_path)

            for image in images:

                text += pytesseract.image_to_string(image)

    except Exception as e:

        print("Error:", pdf_path, e)

    return text

import re

import re

def extract_candidate_name(text, fallback):

    if not text:
        return fallback

    bad_words = {
        "resume",
        "curriculum vitae",
        "experience",
        "objective",
        "profile",
        "professional summary",
        "development tools",
        "education",
        "skills",
        "email",
        "phone",
        "contact"
    }

    lines = text.split("\n")

    for line in lines[:20]:

        line = line.strip()

        if len(line) < 3 or len(line) > 40:
            continue

        lower = line.lower()

        if any(word in lower for word in bad_words):
            continue

        if "@" in line:
            continue

        if any(ch.isdigit() for ch in line):
            continue

        words = line.split()

        if 2 <= len(words) <= 4:
            return line.title()

    return fallback

In [7]:
# cell 6
resume_data = []

candidate_id = 1
limit = 50

for root, dirs, files in os.walk(resume_folder):

    for file in files:

        if candidate_id > limit:
            break

        if file.lower().endswith(".pdf"):

            path = os.path.join(root, file)

            text = extract_text_from_pdf(path)

            resume_data.append({
                "candidate_id": candidate_id,
                "candidate_name": extract_candidate_name(text,
file.replace(".pdf", "")
),
                "resume_text": text
            })

            candidate_id += 1

    if candidate_id > limit:
        break

print("Loaded:", len(resume_data))

Loaded: 50


In [8]:
# cell 7
df = pd.DataFrame(resume_data)

print(df.shape)

df.head()

(50, 3)


,candidate_id,candidate_name,resume_text
0,1,Key Results Areas:,we\n\nKey Results Areas:\n‘© Monttoring the ge...
1,2,Noah Ferry,EXPERIENCE\n11/2019 — present\n\n06/2017 — 08/...
2,3,Robert Smith,\n\nROBERT SMITH\n\nSap Abap Developer III\ni...
3,4,Lydia Botsford,EXPERIENCE\n\nEDUCATION\n\nSKILLS\n\nLydia Bot...
4,5,Kevin Graham,KEVIN GRAHAM\n\nIT/SAP PROFESSIONAL\n\n= 0 9 i...


In [114]:
# cell 8
job_description = """
Human Resources Manager

Skills:
Recruitment
Payroll
Employee Relations
Performance Management
Talent Acquisition
HR Policies
"""

In [115]:
# cell 9
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
# cell 10
resume_embeddings = model.encode(
    df["resume_text"].tolist(),
    show_progress_bar=True
)

jd_embedding = model.encode(job_description)

scores = cosine_similarity(
    [jd_embedding],
    resume_embeddings
)[0]

df["semantic_score"] = scores * 100

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [127]:
# cell 11
skills_list = [
    "python",
    "machine learning",
    "deep learning",
    "tensorflow",
    "pytorch",
    "nlp",
    "sql",
    "aws",
    "docker",
    "git",
    "java",
    "excel",
    "power bi",
    "data analysis"
]

In [128]:
# cell 12
def extract_skills(text):

    text = str(text).lower()

    return [
        skill
        for skill in skills_list
        if skill in text
    ]

In [129]:
# cell 13
jd_skills = extract_skills(job_description)

df["matched_skills"] = df["resume_text"].apply(extract_skills)

df["skill_score"] = (
    df["matched_skills"]
    .apply(lambda x: len(x))
    / max(len(jd_skills), 1)
) * 100

In [130]:
# Cell 14 - Hybrid Score

df["score"] = (
    0.8 * df["semantic_score"]
    + 0.2 * df["skill_score"]
)

print(df[["semantic_score", "skill_score", "score"]].head())

   semantic_score  skill_score      score
0       23.493668        300.0  78.794935
1       21.188303        300.0  76.950644
2       30.996572        200.0  64.797258
3       30.596245        200.0  64.476995
4       27.103065        200.0  61.682453


In [131]:
# Cell 15 - Recommendation

def get_recommendation(score):

    if score >= 75:
        return "Strong Fit"

    elif score >= 55:
        return "Good Fit"

    elif score >= 35:
        return "Potential Fit"

    else:
        return "Weak Fit"

df["recommendation"] = df["score"].apply(get_recommendation)

In [132]:
# cell 16
df["missing_or_weak_signals"] = df[
    "matched_skills"
].apply(
    lambda x: list(set(jd_skills) - set(x))
)

df["evidence"] = df["matched_skills"]

In [133]:
# cell 17
df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

In [134]:
# cell 18
final_df = df[
    [
        "rank",
        "candidate_id",
        "candidate_name",
        "score",
        "recommendation",
        "matched_skills",
        "missing_or_weak_signals",
        "evidence"
    ]
]

final_df = final_df.copy()
final_df["score"] = final_df["score"].round(2)

final_df.head(20)

,rank,candidate_id,candidate_name,score,recommendation,matched_skills,missing_or_weak_signals,evidence
0,1,42,Image_29,78.79,Strong Fit,"[sql, excel, data analysis]",[],"[sql, excel, data analysis]"
1,2,22,Trinity Ankunding,76.95,Strong Fit,"[sql, java, excel]",[],"[sql, java, excel]"
2,3,15,Reilly Aufderhar,64.80,Good Fit,"[sql, excel]",[],"[sql, excel]"
3,4,2,Noah Ferry,64.48,Good Fit,"[java, excel]",[],"[java, excel]"
4,5,21,Kendal Deckow,61.68,Good Fit,"[sql, java]",[],"[sql, java]"
5,6,19,Jan Greenfelder,55.61,Good Fit,"[sql, git]",[],"[sql, git]"
6,7,45,Natalia Pfeffer,49.61,Potential Fit,[excel],[],[excel]
7,8,5,Kevin Graham,49.49,Potential Fit,[sql],[],[sql]
8,9,28,Adrian Penaloza,49.00,Potential Fit,[excel],[],[excel]
9,10,18,Anil Kumar Patra,48.53,Potential Fit,[excel],[],[excel]
